# 0. Lokales Setup (VS Code)


In [2]:
# Zelle 0: Lokale Projektkonfiguration für VS Code / Jupyter

import os

# Feste Projekt- und Datenpfade
PROJECT_ROOT = r"C:\ML_Projekt"
DATA_DIR = r"C:\ML_Projekt\data"

# Hier kannst du einstellen, wo das beste Modell gespeichert wird
CHECKPOINT_DIR = r"C:\ML_Projekt\model"

# Hier kannst du einstellen, wie die neue Best-Model-Datei heißen soll
BEST_MODEL_NAME = "best_model_run4_resnet50.pth"

# Vollständiger Checkpoint-Pfad
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, BEST_MODEL_NAME)

# Checkpoint-Ordner erstellen, falls er noch nicht existiert
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"PROJECT_ROOT:    {PROJECT_ROOT}")
print(f"DATA_DIR:        {DATA_DIR}")
print(f"CHECKPOINT_DIR:  {CHECKPOINT_DIR}")
print(f"BEST_MODEL_NAME: {BEST_MODEL_NAME}")
print(f"CHECKPOINT_PATH: {CHECKPOINT_PATH}")

PROJECT_ROOT:    C:\ML_Projekt
DATA_DIR:        C:\ML_Projekt\data
CHECKPOINT_DIR:  C:\ML_Projekt\model
BEST_MODEL_NAME: best_model_run4_resnet50.pth
CHECKPOINT_PATH: C:\ML_Projekt\model\best_model_run4_resnet50.pth


# 1. Bibliotheken importieren

In [3]:
# Imports

import torch
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

In [4]:
# Reproduzierbarkeit & Device

import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# GPU nutzen falls verfügbar - schneller damit das Training und die Inferenzsind, ansonsten CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Seeds gesetzt: {SEED}")
print(f"Device: {device}") # gibt aus ob CPU oder GPU

Seeds gesetzt: 42
Device: cpu


# 2. Datensatz prüfen

In [5]:
# Datensatz prüfen

# DATA_DIR wird im lokalen Setup definiert.

#TODO: datenstruktur überprüfen -sinnvoll für andere wenn andere bezeichnung der ordner
# expected_classes = ["cubism", "pop_art", "realism"]
# for split in ["train", "val", "test"]:
 #   split_path = os.path.join(DATA_DIR, split)
 #   assert os.path.exists(split_path), f"FEHLER: {split} Ordner fehlt!"
 #   for cls in expected_classes:
 #       cls_path = os.path.join(split_path, cls)
 #       assert os.path.exists(cls_path), f"FEHLER: Klasse {cls} fehlt in {split}!"
#print("Datenstruktur OK ✓")

# Klassen und Bildanzahl prüfen
for split in ["train", "val", "test"]:
    split_path = os.path.join(DATA_DIR, split)
    print(f"\n{split.upper()}:")
    for class_name in sorted(os.listdir(split_path)):
        class_path = os.path.join(split_path, class_name)
        if os.path.isdir(class_path):
            count = len([
              f for f in os.listdir(class_path)
              if f.lower().endswith((".jpg", ".jpeg", ".png"))
            ])
            print(f"  {class_name}: {count} Bilder")


TRAIN:
  baroque: 1050 Bilder
  cubism: 1050 Bilder
  minimalism: 935 Bilder
  pop_art: 1038 Bilder
  realism: 1050 Bilder
  romanticism: 1050 Bilder

VAL:
  baroque: 225 Bilder
  cubism: 225 Bilder
  minimalism: 200 Bilder
  pop_art: 222 Bilder
  realism: 225 Bilder
  romanticism: 225 Bilder

TEST:
  baroque: 225 Bilder
  cubism: 225 Bilder
  minimalism: 202 Bilder
  pop_art: 223 Bilder
  realism: 225 Bilder
  romanticism: 225 Bilder


# 3. Preprocessing definieren

In [6]:
# Preprocessing definieren

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transforms definiert:")
print(f"  Training:  {len(train_transform.transforms)} Schritte")
print(f"  Val/Test:  {len(val_test_transform.transforms)} Schritte")

Transforms definiert:
  Training:  6 Schritte
  Val/Test:  3 Schritte


# 4. Daten laden

In [7]:

# Datensatz einlesen

train_dataset = datasets.ImageFolder(
    root=os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=os.path.join(DATA_DIR, "val"),
    transform=val_test_transform
)

test_dataset = datasets.ImageFolder(
    root=os.path.join(DATA_DIR, "test"),
    transform=val_test_transform

)

# In Batches aufteilen

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Prüfen

print(f"Klassen: {train_dataset.classes}")
print(f"Klassen-Index: {train_dataset.class_to_idx}")
print(f"\nTraining: {len(train_dataset)} Bilder")
print(f"Validation: {len(val_dataset)} Bilder")
print(f"Test: {len(test_dataset)} Bilder")
print(f"\nBatches im Training: {len(train_loader)}")


Klassen: ['baroque', 'cubism', 'minimalism', 'pop_art', 'realism', 'romanticism']
Klassen-Index: {'baroque': 0, 'cubism': 1, 'minimalism': 2, 'pop_art': 3, 'realism': 4, 'romanticism': 5}

Training: 6173 Bilder
Validation: 1322 Bilder
Test: 1325 Bilder

Batches im Training: 97


# 5. Modell aufbauen

In [8]:
model = models.resnet50(weights="IMAGENET1K_V1")

# Alle Layer einfrieren – Layer4 bleibt diesmal eingefroren
for param in model.parameters():
    param.requires_grad = False

# ← Layer4 wird nicht mehr aufgetaut

# Nur fc auftauen
for param in model.fc.parameters():
    param.requires_grad = True

# Letzten Layer ersetzen – Dropout bleibt, 2048 statt 512
num_classes = len(train_dataset.classes)
model.fc = torch.nn.Sequential(
    torch.nn.Dropout(0.4),
    torch.nn.Linear(2048, num_classes)
)

print(f"Modell geladen: ResNet50")
print(f"Anzahl Klassen: {num_classes}")
print(f"Klassen: {train_dataset.classes}")
print(f"\nLetzter Layer: {model.fc}")

model = model.to(device)
print(f"Modell läuft auf: {next(model.parameters()).device}")

0.1%

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\Valentin/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100.0%


Modell geladen: ResNet50
Anzahl Klassen: 6
Klassen: ['baroque', 'cubism', 'minimalism', 'pop_art', 'realism', 'romanticism']

Letzter Layer: Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=2048, out_features=6, bias=True)
)
Modell läuft auf: cpu


# 6. Training konfigurieren

In [9]:
NUM_EPOCHS = 50
LEARNING_RATE = 0.001
PATIENCE = 10

criterion = torch.nn.CrossEntropyLoss()

# ← Nur noch fc im Optimizer, keine zwei Lernraten mehr
optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=LEARNING_RATE
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=3,
    factor=0.5,
)

print(f"Loss-Funktion: CrossEntropyLoss")
print(f"Optimizer: Adam (fc lr={LEARNING_RATE})")
print(f"Epochen: {NUM_EPOCHS}")
print(f"Early Stopping Patience: {PATIENCE}")
print(f"Scheduler: ReduceLROnPlateau (patience=3, factor=0.5)")

Loss-Funktion: CrossEntropyLoss
Optimizer: Adam (fc lr=0.001)
Epochen: 50
Early Stopping Patience: 10
Scheduler: ReduceLROnPlateau (patience=3, factor=0.5)


# 7. Trainingsschleife

In [10]:
torch.manual_seed(SEED)   # Seed festsetzen (Reproduzierbarkeit von Zelle 1)
train_losses = []
val_losses = []
best_val_loss = float('inf')
patience_counter = 0                        # ← neu

for epoch in range(NUM_EPOCHS):

    # ── Training ──────────────────────────────────────────
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ── Validation ────────────────────────────────────────
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    # ── Ausgabe & Checkpoint ──────────────────────────────
    print(f"Epoche {epoch+1:02}/{NUM_EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss:   {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0                            # neu: patience zurücksetzen
        torch.save({                                    # neu: Checkpoint Speicherung angepasst
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "val_loss": avg_val_loss,
        }, CHECKPOINT_PATH)

        print(f"  → Checkpoint gespeichert!")
    else:
        patience_counter += 1                           # ← neu: patience hochzählen
        print(f"  → Keine Verbesserung ({patience_counter}/{PATIENCE})")

    if patience_counter >= PATIENCE:        # ← neu: stoppen
        print(f"\nEarly Stopping nach Epoche {epoch+1}!")
        break

    scheduler.step(avg_val_loss)                                                     # Scheduler Veränderungen

print("\nTraining abgeschlossen!")

KeyboardInterrupt: 

# 8. Kurven plotten

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(train_losses, label="Training Loss", color="blue")
plt.plot(val_losses, label="Validation Loss", color="green")

plt.xlabel("Epoche")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.grid(True)

# Besten Checkpoint markieren

best_epoch = val_losses.index(min(val_losses))
plt.axvline(x=best_epoch, color="red", linestyle="--", label=f"Bester Checkpoint (Epoche {best_epoch+1})")

plt.tight_layout()
plt.show()

print(f"\nBester Val Loss: {min(val_losses):.4f} bei Epoche {best_epoch+1}")


# 9. Test-Evaluation

In [ ]:
# Besten Checkpoint laden
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Bestes Modell geladen – Epoche {checkpoint['epoch']+1}, "
      f"Val Loss: {checkpoint['val_loss']:.4f}")

model.eval()

correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        # Accuracy
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Metriken
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = correct / total * 100
print(f"Test Accuracy: {accuracy:.2f}%")
print(f"Richtig:       {correct} von {total} Bildern")

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print(classification_report(all_labels, all_preds,
    target_names=train_dataset.classes))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=train_dataset.classes,
            yticklabels=train_dataset.classes,
            cmap="Blues")
plt.xlabel("Vorhergesagt")
plt.ylabel("Tatsächlich")
plt.title("Confusion Matrix – Test-Set")
plt.tight_layout()
plt.show()

In [ ]:
print(f"Device von model: {next(model.parameters()).device}")